# VPA Demo: Direct Tool Runs

This notebook runs the lightweight direct-tool path from `docs/use_tools.md`.
It executes CPU-safe tools by default, writes outputs under `/tmp`, and generates
an HTML report. Heavy GPU examples are printed unless `VPA_DEMO_RUN_HEAVY=1` is
set.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for path in [here, *here.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src" / "vibeproteinagent").is_dir():
            return path
    return here


REPO_ROOT = find_repo_root()
RUN_INSTALL = os.environ.get("VPA_DEMO_RUN_INSTALL", "0") == "1"
RUN_HEAVY = os.environ.get("VPA_DEMO_RUN_HEAVY", "0") == "1"
RUN_LLM = os.environ.get("VPA_DEMO_RUN_LLM", "0") == "1"
WORKSPACE_PREFIX = os.environ.get("VPA_DEMO_WORKSPACE_PREFIX", f"vpa_demo_{os.getpid()}")
OUT_DIR = Path(os.environ.get("VPA_DEMO_OUT_DIR", "/tmp")) / WORKSPACE_PREFIX
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep the demo daemon/socket isolated from any VPA daemon the user may already be running.
os.environ.setdefault("VPA_RUNTIME_DIR", str(OUT_DIR / "runtime"))
Path(os.environ["VPA_RUNTIME_DIR"]).mkdir(parents=True, exist_ok=True)


def _candidate_vpa_commands() -> list[list[str]]:
    commands: list[list[str]] = []
    installed = shutil.which("vpa")
    if installed:
        commands.append([installed])
    if (REPO_ROOT / "pyproject.toml").is_file() and shutil.which("uv"):
        commands.append(["uv", "run", "vpa"])
    return commands


def choose_vpa() -> list[str]:
    errors = []
    for base in _candidate_vpa_commands():
        proc = subprocess.run(
            [*base, "version", "--json"],
            cwd=REPO_ROOT,
            text=True,
            capture_output=True,
            env=os.environ.copy(),
            timeout=120,
        )
        if proc.returncode == 0:
            print("Using VPA command:", " ".join(shlex.quote(part) for part in base))
            print(proc.stdout.strip())
            return base
        errors.append((base, proc.returncode, proc.stderr.strip() or proc.stdout.strip()))
    raise RuntimeError(f"Could not find a working vpa command: {errors}")


VPA = choose_vpa()


def show_command(parts: list[str]) -> None:
    print("$ " + " ".join(shlex.quote(str(part)) for part in parts))


def run(parts: list[str], *, check: bool = True, timeout: int = 300, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    parts = [str(part) for part in parts]
    show_command(parts)
    proc = subprocess.run(
        parts,
        cwd=cwd or REPO_ROOT,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, parts, proc.stdout, proc.stderr)
    return proc


def run_json(parts: list[str], *, timeout: int = 300, check: bool = True) -> dict:
    proc = run([*parts, "--json"], timeout=timeout, check=check)
    if proc.returncode != 0:
        return {}
    return json.loads(proc.stdout)


def stop_demo_daemon() -> None:
    run([*VPA, "daemon", "stop", "--workspace", f"{WORKSPACE_PREFIX}_cleanup", "--json"], check=False, timeout=60)


print("Repo root:", REPO_ROOT)
print("Output directory:", OUT_DIR)
print("Runtime directory:", os.environ["VPA_RUNTIME_DIR"])
print("RUN_INSTALL=", RUN_INSTALL, "RUN_HEAVY=", RUN_HEAVY, "RUN_LLM=", RUN_LLM)

## Optional Install

Set `VPA_DEMO_RUN_INSTALL=1` to let the notebook bootstrap the base profile.
The default is to use the VPA runtime already available in your shell or the
repo's `uv run vpa` fallback.

In [ ]:
install_cmd = ["npx", "vibeproteinagent", "install", "--profile", "base", "--cuda-profile", "auto", "--yes"]
if RUN_INSTALL:
    run(install_cmd, timeout=3600)
else:
    print("Skipping install. To run it, set VPA_DEMO_RUN_INSTALL=1.")
    show_command(install_cmd)

## 1. CLI And Tool Surface

These commands should work before any heavy model is installed.

In [ ]:
run([*VPA, "--help"], timeout=120)
run_json([*VPA, "tools"], timeout=120)
for tool in ["sequence_analysis", "protein_qc"]:
    run_json([*VPA, "tool", "info", tool], timeout=120)

## 2. Start An Isolated CPU Worker

The notebook sets a private `VPA_RUNTIME_DIR`, so this daemon does not collide
with another VPA daemon you may be running in a different shell.

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_direct_tools"
stop_demo_daemon()
run_json([*VPA, "daemon", "start", "--cpu-workers", "1", "--workspace", workspace], timeout=120)
run_json([*VPA, "daemon", "status", "--workspace", workspace], timeout=60)

## 3. Run Sequence Analysis And QC

Each `vpa run --json` returns a job id. `vpa wait` waits for completion, and
`vpa results` returns the structured adapter output.

In [ ]:
sequence = "ACDEFGHIKLMNPQRSTVWY"
analysis = run_json([*VPA, "run", "sequence_analysis", f"sequence={sequence}", "--workspace", workspace], timeout=120)
analysis_job = analysis["data"]["job_id"]
run_json([*VPA, "wait", analysis_job, "--workspace", workspace, "--timeout-ms", "30000"], timeout=60)
analysis_results = run_json([*VPA, "results", analysis_job, "--workspace", workspace], timeout=60)
print("Sequence length:", analysis_results["data"]["output"]["sequence_length"])

qc = run_json([
    *VPA,
    "run",
    "protein_qc",
    f"sequence={sequence}",
    'metrics=["gravy","instability_index","pi","polybasic"]',
    "--workspace",
    workspace,
], timeout=120)
qc_job = qc["data"]["job_id"]
run_json([*VPA, "wait", qc_job, "--workspace", workspace, "--timeout-ms", "30000"], timeout=60)
qc_results = run_json([*VPA, "results", qc_job, "--workspace", workspace], timeout=60)
print(json.dumps(qc_results["data"]["output"]["metrics"], indent=2))

## 4. Artifacts, Download, And Report

Reports are generated from stored job output; they do not rerun scientific work.

In [ ]:
run_json([*VPA, "status", qc_job, "--workspace", workspace], timeout=60)
run_json([*VPA, "artifacts", qc_job, "--workspace", workspace], timeout=60)
download_dir = OUT_DIR / "downloaded_qc_artifacts"
run_json([*VPA, "download", qc_job, "--workspace", workspace, "--out", str(download_dir)], timeout=60)
report_path = OUT_DIR / "direct_tool_report.html"
report = run_json([*VPA, "report", qc_job, "--workspace", workspace, "--out", str(report_path)], timeout=120)
print("Report:", report["data"]["report_html"])

## 5. Optional Heavy Direct Runs

These commands require installed GPU tools and a GPU worker. They are skipped by
default.

In [ ]:
heavy_workspace = f"{WORKSPACE_PREFIX}_heavy_tools"
heavy_cmds = [
    [*VPA, "install", "doctor", "--tools", "proteinmpnn,protenix_v2"],
    [*VPA, "daemon", "start", "--workers", "1", "--cpu-workers", "1", "--workspace", heavy_workspace],
    [*VPA, "run", "proteinmpnn", f"pdb_path={REPO_ROOT / 'tests' / 'fixtures' / '6MRR.pdb'}", "num_designs=2", "seed=17", "--workspace", heavy_workspace, "--follow"],
    [*VPA, "run", "protenix_v2", f"input_json={REPO_ROOT / 'tests' / 'fixtures' / 'protenix_v2_input_tiny.json'}", "--workspace", heavy_workspace, "--follow"],
]
if RUN_HEAVY:
    for cmd in heavy_cmds:
        run(cmd, timeout=7200)
else:
    print("Skipping heavy runs. To execute them, set VPA_DEMO_RUN_HEAVY=1.")
    for cmd in heavy_cmds:
        show_command(cmd)

## 6. Stop The Demo Daemon

In [ ]:
run([*VPA, "daemon", "stop", "--workspace", workspace, "--json"], check=False, timeout=60)